In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
import pandas as pd
import numpy as np
import json
import re
import os
from copy import deepcopy

base_csv = "/content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Keystroke_Dynamics/keystroke_intial.csv"

df = pd.read_csv(
    base_csv,
    sep=";",
    engine="python"
)

print(df.shape)
print(df.columns.tolist())
print(df.head())

(117, 11)
['id', 'full_name', 'email', 'session_token', 'consent_given', 'consent_timestamp', 'keystroke_data', 'audio_url', 'video_url', 'completed_at', 'created_at']
                                     id                 full_name  \
0  28f503ad-299f-4a2f-96c7-ab22875e0e18                      Test   
1  f06f8bb5-8303-416b-bf9c-e247eb17b864   Kavinesh Ganeshamoorthy   
2  4cf15e9c-2f70-47cb-803a-52bb527eb74d                   Raabiya   
3  a5ad9b9a-aadb-40b9-8d74-7baf4bb2fb9f            Shahood Irshan   
4  585fb1cb-bb40-44ae-85c5-498c971c3436  Madhumitha Balajeyandhan   

                         email              session_token  consent_given  \
0               test@gmail.com  1769877865550-cngi8444j34           True   
1         kavin.prvt@gmail.com  1770008423822-5ek366c1tyi           True   
2  fathima.20222116.@iit.ac.lk  1770364827761-mr9ug9dhlib           True   
3      irshanshahood@gmail.com  1770016503680-7cdim9wd80u           True   
4            madhumi@gmail.com  17702

In [4]:
def parse_keystrokes(x):
    try:
        return json.loads(x)
    except:
        return None

df["keystrokes_parsed"] = df["keystroke_data"].apply(parse_keystrokes)

# Keep only usable rows
df = df[df["keystrokes_parsed"].notna()].copy()
df = df[df["keystrokes_parsed"].apply(lambda x: isinstance(x, list) and len(x) > 0)].copy()

print("Usable rows:", df.shape)

Usable rows: (117, 12)


In [5]:
def extract_numeric_token(text):
    text = str(text)
    nums = re.findall(r"\d{10,}", text)
    return nums[-1] if nums else None

df["session_token_clean"] = df["session_token"].apply(extract_numeric_token)
print(df[["session_token", "session_token_clean"]].head())

               session_token session_token_clean
0  1769877865550-cngi8444j34       1769877865550
1  1770008423822-5ek366c1tyi       1770008423822
2  1770364827761-mr9ug9dhlib       1770364827761
3  1770016503680-7cdim9wd80u       1770016503680
4  1770206737552-7b84if03p6b       1770206737552


build genuine dataset

In [6]:
genuine_df = df[[
    "id",
    "full_name",
    "email",
    "session_token",
    "session_token_clean",
    "audio_url",
    "video_url",
    "keystrokes_parsed"
]].copy()

genuine_df["attack_type"] = "genuine"
genuine_df["label"] = 0

print(genuine_df.shape)
print(genuine_df.head())

(117, 10)
                                     id                 full_name  \
0  28f503ad-299f-4a2f-96c7-ab22875e0e18                      Test   
1  f06f8bb5-8303-416b-bf9c-e247eb17b864   Kavinesh Ganeshamoorthy   
2  4cf15e9c-2f70-47cb-803a-52bb527eb74d                   Raabiya   
3  a5ad9b9a-aadb-40b9-8d74-7baf4bb2fb9f            Shahood Irshan   
4  585fb1cb-bb40-44ae-85c5-498c971c3436  Madhumitha Balajeyandhan   

                         email              session_token session_token_clean  \
0               test@gmail.com  1769877865550-cngi8444j34       1769877865550   
1         kavin.prvt@gmail.com  1770008423822-5ek366c1tyi       1770008423822   
2  fathima.20222116.@iit.ac.lk  1770364827761-mr9ug9dhlib       1770364827761   
3      irshanshahood@gmail.com  1770016503680-7cdim9wd80u       1770016503680   
4            madhumi@gmail.com  1770206737552-7b84if03p6b       1770206737552   

                                           audio_url  \
0               1769877865550-cn

generate replay attacks

In [7]:
replay_df = genuine_df.copy().reset_index(drop=True)

# Shuffle keystrokes across rows
shuffled = replay_df["keystrokes_parsed"].sample(frac=1, random_state=42).reset_index(drop=True)
replay_df["keystrokes_parsed"] = shuffled

# Keep only rows where the session did not get its original pattern back
same_mask = replay_df["session_token_clean"] == genuine_df["session_token_clean"].reset_index(drop=True)
replay_df = replay_df[~same_mask].copy().reset_index(drop=True)

replay_df["attack_type"] = "replay"
replay_df["label"] = 1

print("Replay rows:", replay_df.shape)

Replay rows: (0, 10)


generate synthetic attacks


In [8]:
def perturb_keystrokes(events,
                       hold_scale_range=(0.75, 1.35),
                       flight_scale_range=(0.70, 1.40),
                       additive_noise_std=8.0):
    events = deepcopy(events)
    new_events = []

    for ev in events:
        ev_new = dict(ev)

        # holdTime
        if ev_new.get("holdTime") is not None:
            h = float(ev_new["holdTime"])
            h = h * np.random.uniform(*hold_scale_range) + np.random.normal(0, additive_noise_std)
            ev_new["holdTime"] = max(0, h)

        # flightTime
        if ev_new.get("flightTime") is not None:
            f = float(ev_new["flightTime"])
            f = f * np.random.uniform(*flight_scale_range) + np.random.normal(0, additive_noise_std)
            ev_new["flightTime"] = max(0, f)

        new_events.append(ev_new)

    return new_events

In [9]:
np.random.seed(42)

synthetic_df = genuine_df.copy().reset_index(drop=True)
synthetic_df["keystrokes_parsed"] = synthetic_df["keystrokes_parsed"].apply(perturb_keystrokes)
synthetic_df["attack_type"] = "synthetic"
synthetic_df["label"] = 1

print("Synthetic rows:", synthetic_df.shape)

Synthetic rows: (117, 10)


combine everything

In [10]:
full_ks_df = pd.concat(
    [genuine_df, replay_df, synthetic_df],
    ignore_index=True
).reset_index(drop=True)

print(full_ks_df.shape)
print(full_ks_df["attack_type"].value_counts())
print(full_ks_df["label"].value_counts())

(234, 10)
attack_type
genuine      117
synthetic    117
Name: count, dtype: int64
label
0    117
1    117
Name: count, dtype: int64


extract keystroke features

In [11]:
def extract_keystroke_features(events):
    hold = [e.get("holdTime") for e in events if e.get("holdTime") is not None]
    flight = [e.get("flightTime") for e in events if e.get("flightTime") is not None]
    keys = [e.get("keyCode") for e in events if e.get("keyCode") is not None]

    hold = np.array(hold, dtype=float) if len(hold) else np.array([])
    flight = np.array(flight, dtype=float) if len(flight) else np.array([])
    keys = np.array(keys, dtype=float) if len(keys) else np.array([])

    def safe_stat(arr, fn, default=0):
        return fn(arr) if len(arr) > 0 else default

    return pd.Series({
        "hold_mean": safe_stat(hold, np.mean),
        "hold_std": safe_stat(hold, np.std),
        "hold_min": safe_stat(hold, np.min),
        "hold_max": safe_stat(hold, np.max),
        "hold_median": safe_stat(hold, np.median),

        "flight_mean": safe_stat(flight, np.mean),
        "flight_std": safe_stat(flight, np.std),
        "flight_min": safe_stat(flight, np.min),
        "flight_max": safe_stat(flight, np.max),
        "flight_median": safe_stat(flight, np.median),

        "num_keys": len(events),
        "unique_keys": len(np.unique(keys)) if len(keys) > 0 else 0,
        "hold_flight_ratio": (np.mean(hold) / np.mean(flight)) if len(hold) > 0 and len(flight) > 0 and np.mean(flight) != 0 else 0
    })

In [12]:
feature_df = full_ks_df["keystrokes_parsed"].apply(extract_keystroke_features)

feat_df = pd.concat([
    full_ks_df[[
        "id",
        "full_name",
        "email",
        "session_token",
        "session_token_clean",
        "audio_url",
        "video_url",
        "attack_type",
        "label"
    ]].reset_index(drop=True),
    feature_df.reset_index(drop=True)
], axis=1)

print(feat_df.shape)
print(feat_df["attack_type"].value_counts())
print(feat_df["label"].value_counts())
print(feat_df.head())

(234, 22)
attack_type
genuine      117
synthetic    117
Name: count, dtype: int64
label
0    117
1    117
Name: count, dtype: int64
                                     id                 full_name  \
0  28f503ad-299f-4a2f-96c7-ab22875e0e18                      Test   
1  f06f8bb5-8303-416b-bf9c-e247eb17b864   Kavinesh Ganeshamoorthy   
2  4cf15e9c-2f70-47cb-803a-52bb527eb74d                   Raabiya   
3  a5ad9b9a-aadb-40b9-8d74-7baf4bb2fb9f            Shahood Irshan   
4  585fb1cb-bb40-44ae-85c5-498c971c3436  Madhumitha Balajeyandhan   

                         email              session_token session_token_clean  \
0               test@gmail.com  1769877865550-cngi8444j34       1769877865550   
1         kavin.prvt@gmail.com  1770008423822-5ek366c1tyi       1770008423822   
2  fathima.20222116.@iit.ac.lk  1770364827761-mr9ug9dhlib       1770364827761   
3      irshanshahood@gmail.com  1770016503680-7cdim9wd80u       1770016503680   
4            madhumi@gmail.com  1770206737552-7b

In [13]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score
from xgboost import XGBClassifier

In [14]:
feature_cols = [
    "hold_mean", "hold_std", "hold_min", "hold_max", "hold_median",
    "flight_mean", "flight_std", "flight_min", "flight_max", "flight_median",
    "num_keys", "unique_keys", "hold_flight_ratio"
]

train_df, val_df = train_test_split(
    feat_df,
    test_size=0.2,
    random_state=42,
    stratify=feat_df["label"]
)

X_train = train_df[feature_cols].copy()
y_train = train_df["label"].astype(int)

X_val = val_df[feature_cols].copy()
y_val = val_df["label"].astype(int)

imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()

X_train = imputer.fit_transform(X_train)
X_val = imputer.transform(X_val)

X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)

model = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_val)
y_prob = model.predict_proba(X_val)[:, 1]

print("Accuracy:", accuracy_score(y_val, y_pred))
print("ROC-AUC:", roc_auc_score(y_val, y_prob))
print("\nClassification Report:")
print(classification_report(y_val, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_pred))

Accuracy: 0.9574468085106383
ROC-AUC: 0.9945652173913044

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.96      0.96        24
           1       0.96      0.96      0.96        23

    accuracy                           0.96        47
   macro avg       0.96      0.96      0.96        47
weighted avg       0.96      0.96      0.96        47


Confusion Matrix:
[[23  1]
 [ 1 22]]


In [15]:
keystroke_val_results = pd.DataFrame({
    "session_token_clean": val_df["session_token_clean"].values,
    "session_token": val_df["session_token"].values,
    "attack_type": val_df["attack_type"].values,
    "true_label": y_val.values,
    "keystroke_pred": y_pred,
    "keystroke_prob_spoof": y_prob
})

save_dir = "/content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Keystroke_model_outputs/"
os.makedirs(save_dir, exist_ok=True)

keystroke_val_results.to_csv(save_dir + "keystroke_val_predictions.csv", index=False)
print("Saved:", save_dir + "keystroke_val_predictions.csv")
print(keystroke_val_results.head())

Saved: /content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Keystroke_model_outputs/keystroke_val_predictions.csv
  session_token_clean              session_token attack_type  true_label  \
0       1770008423822  1770008423822-5ek366c1tyi     genuine           0   
1       1770583681000   1770583681000-kkbdy2g8w6   synthetic           1   
2       1770557958937   1770557958937-o7zm1xx7z5     genuine           0   
3       1771426980393  1771426980393-nzuzo8hi3mr   synthetic           1   
4       1770259504525  1770259504525-yq212ny4ozl   synthetic           1   

   keystroke_pred  keystroke_prob_spoof  
0               0              0.004969  
1               1              0.984553  
2               0              0.004291  
3               1              0.984008  
4               1              0.973028  


In [16]:
import joblib, os

save_dir = "/content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Models"
os.makedirs(save_dir, exist_ok=True)

# Save all three — you need all of them for inference
joblib.dump(model,   f"{save_dir}/xgb_keystroke.pkl")
joblib.dump(imputer, f"{save_dir}/keystroke_imputer.pkl")
joblib.dump(scaler,  f"{save_dir}/keystroke_scaler.pkl")

print("Keystroke model saved successfully.")
print(f"Location: {save_dir}")

Keystroke model saved successfully.
Location: /content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Models
